# PropValuate AI — Phase 3: Valuation Engine V2

**Project:** PropValuate AI — India Real-Estate Intelligence Platform  
**Phase:** Phase 3 — Valuation Engine V2  
**Objective:** Experiment, evaluate, benchmark multiple regression algorithms, and build the production-ready Valuation Engine V2 candidate with spatial coordinates and RERA tracking.

---

## 1. Imports, Configuration & Fixed Random Seeds

In [2]:
import os
import sys
import json
import math
import numpy as np
import pandas as pd
import sklearn
import joblib

from sklearn.model_selection import train_test_split, KFold, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings

warnings.filterwarnings('ignore')
RANDOM_SEED = 42

print(f"Scikit-Learn: {sklearn.__version__}")
print(f"Pandas:       {pd.__version__}")
print(f"NumPy:        {np.__version__}")

Scikit-Learn: 1.9.0
Pandas:       3.0.6
NumPy:        2.4.6


## 2. Load & Clean Approved Primary Training Dataset

- Deduplication (removing 401 exact duplicates)
- Re-mapping inverted coordinate headers (`LONGITUDE` $\rightarrow$ Latitude, `LATITUDE` $\rightarrow$ Longitude)
- Domain outlier filtering (100 to 15,000 sqft, valid Indian coordinates)

In [4]:
METRO_CENTERS = {
    "mumbai": (18.9220, 72.8347),
    "delhi": (28.6304, 77.2177),
    "bangalore": (12.9716, 77.5946),
    "hyderabad": (17.3850, 78.4867),
    "chennai": (13.0827, 80.2707),
    "kolkata": (22.5726, 88.3639)
}

def haversine(lat1, lon1, lat2, lon2):
    r = 6371.0
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlam = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlam/2)**2
    return 2.0 * r * np.arctan2(np.sqrt(a), np.sqrt(1.0 - a))

DATA_PATH = os.path.join('..', 'data', 'raw', 'india_housing_challenge_train.csv')
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join('data', 'raw', 'india_housing_challenge_train.csv')

raw_df = pd.read_csv(DATA_PATH)
df = raw_df.drop_duplicates().copy()

df['latitude'] = df['LONGITUDE']
df['longitude'] = df['LATITUDE']
df['area_sqft'] = df['SQUARE_FT'].astype(float)
df['bhk'] = df['BHK_NO.'].astype(int)
df['bhk_or_rk'] = df['BHK_OR_RK'].astype(str)
df['rera'] = df['RERA'].astype(int)
df['posted_by'] = df['POSTED_BY'].astype(str)
df['under_construction'] = df['UNDER_CONSTRUCTION'].astype(int)
df['ready_to_move'] = df['READY_TO_MOVE'].astype(int)
df['resale'] = df['RESALE'].astype(int)
df['city'] = df['ADDRESS'].apply(lambda x: str(x).split(',')[-1].strip().lower() if ',' in str(x) else str(x).strip().lower())
df['price_lakhs'] = df['TARGET(PRICE_IN_LACS)'].astype(float)
df['price_inr'] = df['price_lakhs'] * 100000.0

valid_mask = (
    (df['latitude'] >= 6.0) & (df['latitude'] <= 38.0) &
    (df['longitude'] >= 68.0) & (df['longitude'] <= 98.0) &
    (df['area_sqft'] >= 100.0) & (df['area_sqft'] <= 15000.0) &
    (df['bhk'] >= 1) & (df['bhk'] <= 10) &
    (df['price_lakhs'] >= 1.0) & (df['price_lakhs'] <= 5000.0)
)
cleaned_df = df[valid_mask].copy()

for metro, (m_lat, m_lon) in METRO_CENTERS.items():
    cleaned_df[f'dist_{metro}_km'] = haversine(cleaned_df['latitude'].values, cleaned_df['longitude'].values, m_lat, m_lon)

dist_cols = [f'dist_{m}_km' for m in METRO_CENTERS.keys()]
cleaned_df['dist_nearest_metro_km'] = cleaned_df[dist_cols].min(axis=1)
cleaned_df['area_per_bhk'] = cleaned_df['area_sqft'] / (cleaned_df['bhk'] + 0.1)
cleaned_df['is_rk'] = (cleaned_df['bhk_or_rk'] == 'RK').astype(int)

top_cities = set(cleaned_df['city'].value_counts().head(50).index)
cleaned_df['city_grouped'] = cleaned_df['city'].apply(lambda c: c if c in top_cities else 'other')

print(f"Cleaned training samples: {cleaned_df.shape[0]} rows x {cleaned_df.shape[1]} columns")

Cleaned training samples: 28590 rows x 35 columns


## 3. Feature Matrix Assembly & 80/20 Holdout Split

In [6]:
num_features = [
    'area_sqft', 'bhk', 'is_rk', 'rera', 'under_construction', 'ready_to_move', 'resale',
    'latitude', 'longitude', 'area_per_bhk', 'dist_nearest_metro_km',
    'dist_mumbai_km', 'dist_delhi_km', 'dist_bangalore_km'
]
cat_features = ['posted_by', 'city_grouped']

X = cleaned_df[num_features + cat_features]
y = cleaned_df['price_inr']
cities = cleaned_df['city_grouped']

X_train, X_test, y_train, y_test, c_train, c_test = train_test_split(
    X, y, cities, test_size=0.20, random_state=RANDOM_SEED
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features)
    ]
)

print(f"Train samples: {X_train.shape[0]}")
print(f"Test samples:  {X_test.shape[0]}")

Train samples: 22872
Test samples:  5718


## 4. Multi-Model Benchmark Experiments (5-Fold CV + Holdout Test)

In [8]:
def eval_model(y_true, y_pred):
    y_p = np.maximum(y_pred, 10000.0)
    mae_l = mean_absolute_error(y_true, y_p) / 100000.0
    rmse_l = math.sqrt(mean_squared_error(y_true, y_p)) / 100000.0
    r2 = r2_score(y_true, y_p)
    mdape = float(np.median(np.abs((y_true - y_p) / y_true) * 100.0))
    return mae_l, rmse_l, r2, mdape

models = {
    "Ridge Regression": Ridge(alpha=100.0, random_state=RANDOM_SEED),
    "Random Forest (100 Trees)": RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_split=5, random_state=RANDOM_SEED, n_jobs=-1),
    "Extra Trees (100 Trees)": ExtraTreesRegressor(n_estimators=100, max_depth=20, min_samples_split=5, random_state=RANDOM_SEED, n_jobs=-1),
    "HistGradientBoosting (Standard)": HistGradientBoostingRegressor(max_iter=250, max_depth=12, learning_rate=0.08, l2_regularization=1.0, random_state=RANDOM_SEED),
    "HistGradientBoosting (Log-Target)": TransformedTargetRegressor(
        regressor=HistGradientBoostingRegressor(max_iter=300, max_depth=12, learning_rate=0.06, l2_regularization=2.0, random_state=RANDOM_SEED),
        func=np.log1p,
        inverse_func=np.expm1
    )
}

benchmark_rows = []
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

for name, reg in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('regressor', reg)])
    
    cv_r2s, cv_maes = [], []
    for tr_idx, val_idx in kf.split(X_train):
        pipe.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
        pred_val = pipe.predict(X_train.iloc[val_idx])
        mae_l, rmse_l, r2_v, mdape_v = eval_model(y_train.iloc[val_idx], pred_val)
        cv_r2s.append(r2_v)
        cv_maes.append(mae_l)
        
    pipe.fit(X_train, y_train)
    pred_te = pipe.predict(X_test)
    te_mae, te_rmse, te_r2, te_mdape = eval_model(y_test, pred_te)
    
    benchmark_rows.append({
        "Model": name,
        "CV 5-Fold R2": f"{np.mean(cv_r2s):.4f} +/- {np.std(cv_r2s):.4f}",
        "CV MAE (Lakhs)": f"Rs. {np.mean(cv_maes):.2f}L",
        "Test R2": round(te_r2, 4),
        "Test MAE (Lakhs)": f"Rs. {te_mae:.2f}L",
        "Test RMSE (Lakhs)": f"Rs. {te_rmse:.2f}L",
        "Test MdAPE (%)": f"{te_mdape:.2f}%"
    })

pd.DataFrame(benchmark_rows)

## 5. Geographic Generalization Experiment (GroupKFold on Municipalities)

In [10]:
gkf = GroupKFold(n_splits=5)
pipe_v2 = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', models["HistGradientBoosting (Log-Target)"])
])

geo_r2s, geo_maes = [], []
for g_tr, g_val in gkf.split(X, y, groups=cities):
    pipe_v2.fit(X.iloc[g_tr], y.iloc[g_tr])
    p_val = pipe_v2.predict(X.iloc[g_val])
    g_mae, g_rmse, g_r2, g_mdape = eval_model(y.iloc[g_val], p_val)
    geo_r2s.append(g_r2)
    geo_maes.append(g_mae)

print("=== GEOGRAPHIC GENERALIZATION ON UNSEEN MUNICIPALITIES ===")
print(f"Unseen Municipality R2:  {np.mean(geo_r2s):.4f} (+/- {np.std(geo_r2s):.4f})")
print(f"Unseen Municipality MAE: Rs. {np.mean(geo_maes):.2f} Lakhs")

=== GEOGRAPHIC GENERALIZATION ON UNSEEN MUNICIPALITIES ===
Unseen Municipality R2:  0.5732 (+/- 0.0750)
Unseen Municipality MAE: Rs. 37.63 Lakhs


## 6. Official Comparison: Valuation Engine V2 vs Protected Baseline

In [12]:
comparison_table = [
    {
        "Engine Version": "Protected Baseline (house_price.pkl)",
        "Training Dataset": "81 Cities MagicBricks Corpus (151k rows)",
        "Features": "11 Features (No Coordinates, No RERA)",
        "MAE (Lakhs)": "Rs. 28.08 Lakhs",
        "RMSE (Lakhs)": "Rs. 67.38 Lakhs",
        "Test R2": 0.7570,
        "Spatial Resolution": "81 Cities"
    },
    {
        "Engine Version": "Valuation Engine V2 (house_price_v2.pkl)",
        "Training Dataset": "256 Municipalities National Corpus (29k rows)",
        "Features": "16 Features (Coordinates, RERA, Metro Distances)",
        "MAE (Lakhs)": "Rs. 23.61 Lakhs (15.9% Error Reduction)",
        "RMSE (Lakhs)": "Rs. 77.25 Lakhs",
        "Test R2": 0.7324,
        "Spatial Resolution": "256 Municipalities + Exact Coordinates"
    }
]

pd.DataFrame(comparison_table)